# PARP cleaning debug notebook
Step through the parsing so we can see exactly which row becomes the header and how dates are built.
Run cells in order.

In [1]:
from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("../PARP")
files = sorted([*INPUT_ROOT.rglob("*.xls"), *INPUT_ROOT.rglob("*.xlsx")])
print("files", len(files))
files[:5]

files 12


[WindowsPath('../PARP/2001/01 JAN 2001 PARP REPORTS.xls'),
 WindowsPath('../PARP/2001/02 FEB 2001 PARP REPORTS.xls'),
 WindowsPath('../PARP/2001/03 MAR 2001 PARP REPORTS.xls'),
 WindowsPath('../PARP/2001/04 APR 2001 PARP REPORTS.xls'),
 WindowsPath('../PARP/2001/05 MAY 2001 PARP REPORTS.xls')]

In [2]:
path = files[0] if files else None
path

WindowsPath('../PARP/2001/01 JAN 2001 PARP REPORTS.xls')

In [3]:
engine = "xlrd" if path and path.suffix.lower() == ".xls" else "openpyxl"
xl = pd.ExcelFile(path, engine=engine)
print("sheets", xl.sheet_names)
sheet = xl.sheet_names[0]
df_raw = pd.read_excel(xl, sheet_name=sheet, header=None, dtype=str)
df_raw.head(12)

sheets ['OAPARP25', 'OAPARP30-A', 'OAPARP30-B', 'OAPARP30-C', 'OAPARP35-A', 'OAPARP35-B', 'OAPARP40', 'OAPARP45', 'OAPARP50', 'OAPARP61-A', 'OAPARP61-B', 'OAPARP61-C', 'OAPARP62']


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,NaN,OAPARP25,NaN,NaN,WAHA OIL COMPANY,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,MONTHLY WELL TEST REPORT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,TEST LENGTH IN HOURS,MTD,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HOURS,PCT WTR,NET GOR,WELL REMARK,NaN
5,JANUARY -2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,A-064-32,14,1,1,6,I,...,752,76,,,31,0,9.2,,,
7,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,QQ-001-32,,,,,,...,,,,,0,0,,,SHUT-IN,
8,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,SS-001-32,,,,,,...,,,,,0,0,,,SHUT-IN,
9,2001-01,BAHI,BASS,BAHI SAND STONE,A-001-32,,,,,,...,,,,,0,0,,,NOW SUSPEND,


In [4]:
HEADER_FILTER_COL_INDEX = 3
SKIP_ROWS_AFTER_FILTER = 3

filtered = df_raw[df_raw.iloc[:, HEADER_FILTER_COL_INDEX].notna()]
filtered.head(12)

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
4,CYCLE,FIELD NAME,RESVR,RESERVOIR DETAILED NAME,WELL,TEST DAY,TEST MONTH,TEST YEAR,TEST LENGTH IN HOURS,MTD,...,DAILY OIL BBLS,DAILY WATER BBLS,DAILY NETGAS SMCF,DAILY GLGAS SMCF,PROD DAYS,PROD HOURS,PCT WTR,NET GOR,WELL REMARK,NaN
6,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,A-064-32,14,1,1,6,I,...,752,76,,,31,0,9.2,,,
7,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,QQ-001-32,,,,,,...,,,,,0,0,,,SHUT-IN,
8,2001-01,BAHI,BARG,BAHI-ARGUB SAND / STONE,SS-001-32,,,,,,...,,,,,0,0,,,SHUT-IN,
9,2001-01,BAHI,BASS,BAHI SAND STONE,A-001-32,,,,,,...,,,,,0,0,,,NOW SUSPEND,
10,2001-01,BAHI,BASS,BAHI SAND STONE,A-069-32,25,1,1,6,I,...,144,272,,,12,0,65.4,,,
11,2001-01,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,23,1,1,24,I,...,119,1150,,,30,0,90.6,,,
12,2001-01,BAHI,BAUK,BAHI UPER KALASH,OO-001-32,20,9,,24,I,...,854,0,,,19,0,0,,PREV TEST,
13,2001-01,BAHI,BPL7,PALEOCENE,A-002-32,,,,,,...,,,,,0,0,,,ABANDONED,
14,2001-01,BAHI,BPL7,PALEOCENE,A-003-32,,,,,,...,,,,,0,0,,,ABANDONED,


In [5]:
def locate_header_index(df):
    for idx, row in df.iterrows():
        values = {
            str(v).strip().upper()
            for v in row.tolist()
            if pd.notna(v) and str(v).strip()
        }
        has_field = "FIELD" in values or "FIELD NAME" in values
        has_resvr = "RESVR" in values or "RESVR CODE" in values
        has_date = (
            "TEST YEAR" in values
            or "YR" in values
            or "TEST MONTH" in values
            or "WELL TEST DATE" in values
            or "WELL TEST DAY" in values
        )
        if has_field and has_resvr and has_date:
            return idx
    return None

header_idx = locate_header_index(filtered)
header_idx

4

In [6]:
filtered.loc[header_idx] if header_idx is not None else None

0                       CYCLE
1                  FIELD NAME
2                       RESVR
3     RESERVOIR DETAILED NAME
4                        WELL
5                    TEST DAY
6                  TEST MONTH
7                   TEST YEAR
8        TEST LENGTH IN HOURS
9                         MTD
10                        EXT
11                  PRES HEAD
12                  PSIG SEP`
13                CHOKES PROD
14                   CHOKE GL
15                   GRAV API
16             DAILY OIL BBLS
17           DAILY WATER BBLS
18          DAILY NETGAS SMCF
19           DAILY GLGAS SMCF
20                  PROD DAYS
21                 PROD HOURS
22                    PCT WTR
23                    NET GOR
24                WELL REMARK
25                        NaN
Name: 4, dtype: object

In [7]:
COLUMN_ALIASES = {
    "FIELD NAME": "FIELD",
    "RESVR CODE": "RESVR",
    "RESERVOIR NAME": "RESERVOIR",
    "RESERVOIR DETAILED NAME": "RESERVOIR",
    "WELL NO": "WELL",
    "WELL": "WELL",
    "WELL STATUS": "WELL_1",
    "TEST DAY": "DAY",
    "WELL TEST DAY": "DAY",
    "TEST MONTH": "MTH",
    "TEST YEAR": "YR",
    "TEST LENGTH IN HOURS": "TEST LENGHT IN HOURS",
    "PERC BS&W": "PERC",
    "GTY & 60 F": "GTY &",
    "GRAV API": "GRAVITY",
}
GROUP_HEADERS = {
    "WELL TEST DATE",
    "WELL STATUS",
    "WELL TEST DATA",
    "WELL TEST",
}
SUBHEADER_TOKENS = {
    "MTH",
    "DAY",
    "YR",
    "HRS",
    "CPR",
    "TPR",
    "SEP",
    "BOPD",
    "BWPD",
    "GOR",
}


def normalize_col_name(value):
    text = "" if pd.isna(value) else str(value)
    text = " ".join(text.replace("\n", " ").split())
    return text.strip()


def build_headers(row):
    headers = []
    for idx, value in enumerate(row):
        name = "" if pd.isna(value) else str(value).strip()
        if not name or name.lower() == "nan":
            name = f"Column{idx + 1}"
        headers.append(name)
    return headers


def is_subheader_row(row):
    values = {
        normalize_col_name(v).upper()
        for v in row.tolist()
        if pd.notna(v) and normalize_col_name(v)
    }
    return bool(values & SUBHEADER_TOKENS)


def merge_headers(primary, secondary):
    merged = list(primary)
    for idx, sub in enumerate(secondary):
        sub_clean = normalize_col_name(sub)
        if not sub_clean or sub_clean.lower().startswith("column"):
            continue
        prim_clean = normalize_col_name(primary[idx]).upper()
        if not prim_clean or prim_clean.startswith("COLUMN") or prim_clean in GROUP_HEADERS:
            merged[idx] = sub_clean
    return merged


In [8]:
def normalize_year_part(value):
    if pd.isna(value):
        return value
    text = str(value).strip()
    if not text:
        return text
    try:
        number = int(float(text))
    except ValueError:
        return text
    if number < 100:
        if number <= 29:
            return str(2000 + number)
        return str(1900 + number)
    return str(number)


def find_date_columns(columns):
    candidates = [
        ("MTH", "DAY", "YR"),
        ("MONTH", "DAY", "YEAR"),
    ]
    for month_col, day_col, year_col in candidates:
        if all(col in columns for col in [month_col, day_col, year_col]):
            return month_col, day_col, year_col
    return None


start = filtered.loc[header_idx:].copy() if header_idx is not None else filtered.iloc[SKIP_ROWS_AFTER_FILTER:].copy()
header_row = start.iloc[0]
subheader_row = start.iloc[1] if len(start) > 1 else None

headers = build_headers(header_row)
if subheader_row is not None and is_subheader_row(subheader_row):
    subheaders = build_headers(subheader_row)
    headers = merge_headers(headers, subheaders)
    work = start.iloc[2:].copy()
else:
    work = start.iloc[1:].copy()

normalized = [normalize_col_name(c) for c in headers]
mapped = [COLUMN_ALIASES.get(name, name) for name in normalized]
work.columns = mapped

date_cols = find_date_columns(work.columns)
date_cols

('MTH', 'DAY', 'YR')

In [9]:
if date_cols:
    month_col, day_col, year_col = date_cols
    work[year_col] = work[year_col].apply(normalize_year_part)
    preview = pd.DataFrame({
        "M": work[month_col].head(10),
        "D": work[day_col].head(10),
        "Y": work[year_col].head(10),
    })
    display(preview)
    date_text = (
        work[month_col].fillna("").astype(str).str.strip()
        + "/"
        + work[day_col].fillna("").astype(str).str.strip()
        + "/"
        + work[year_col].fillna("").astype(str).str.strip()
    )
    display(pd.to_datetime(date_text, errors="coerce", dayfirst=True).head(10))

,M,D,Y
6,1,14,2001
7,,,
8,,,
9,,,
10,1,25,2001
11,1,23,2001
12,9,20,
13,,,
14,,,
15,,,


C:\Users\OPS010\AppData\Local\Temp\ipykernel_61512\563531025.py:17: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  display(pd.to_datetime(date_text, errors="coerce", dayfirst=True).head(10))


6    2001-01-14
7           NaT
8           NaT
9           NaT
10   2001-01-25
11   2001-01-23
12          NaT
13          NaT
14          NaT
15          NaT
dtype: datetime64[ns]

In [10]:
import importlib.util

spec = importlib.util.spec_from_file_location("clean_parp_reports", "src/clean_parp_reports.py")
mod = importlib.util.module_from_spec(spec)
module_path = Path("../src/clean_parp_reports.py").resolve()
spec = importlib.util.spec_from_file_location("clean_parp_reports", module_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

one = mod.transform_file(path)
one.head(10)

C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date


,source,FIELD,RESVR,RESERVOIR,WELL,TEST DATE,TEST LENGHT IN HOURS,WELL TYPE,PCT WTR,GRAVITY,64S,PRES HEAD,PSIG SEP,DAILY OIL BBLS,DAILY WATER BBLS,GOR,GOR
11,01 JAN 2001 PARP REPORTS.xls,BAHI,BARG,BAHI-ARGUB SAND / STONE,A-064-32,2001-01-14,<NA>,<NA>,9.2,45.1,0,<NA>,<NA>,<NA>,<NA>,33,...
13,01 JAN 2001 PARP REPORTS.xls,BAHI,BASS,BAHI SAND STONE,A-069-32,2001-01-25,<NA>,<NA>,65.4,44.3,0,<NA>,<NA>,<NA>,<NA>,165,...
15,01 JAN 2001 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,KK-001-32,2001-01-23,<NA>,<NA>,90.6,45.6,0,<NA>,<NA>,<NA>,<NA>,384,...
16,01 JAN 2001 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,OO-001-32,2000-09-20,<NA>,<NA>,0,41.7,0,<NA>,<NA>,<NA>,<NA>,385,...
18,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,2001-01-01,<NA>,<NA>,41.4,45.2,0,<NA>,<NA>,<NA>,<NA>,165,...
19,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-008-32,2001-01-01,<NA>,<NA>,76.7,45.7,0,<NA>,<NA>,<NA>,<NA>,165,...
20,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,2001-01-02,<NA>,<NA>,72.3,44.6,0,<NA>,<NA>,<NA>,<NA>,165,...
21,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-010-32,2000-09-01,<NA>,<NA>,91.9,44.6,0,<NA>,<NA>,<NA>,<NA>,165,...
22,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-012-32,2001-01-02,<NA>,<NA>,89.5,43.3,0,<NA>,<NA>,<NA>,<NA>,165,...
23,01 JAN 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-013-32,2001-01-05,<NA>,<NA>,82.4,46.2,0,<NA>,<NA>,<NA>,<NA>,165,...


In [11]:
one[["TEST DATE"]].head(10)

,TEST DATE
11,2001-01-14
13,2001-01-25
15,2001-01-23
16,2000-09-20
18,2001-01-01
19,2001-01-01
20,2001-01-02
21,2000-09-01
22,2001-01-02
23,2001-01-05


In [13]:
bad_files = []

for p in files:
    cleaned = mod.transform_file(p)
    if cleaned.empty:
        continue
    dates = pd.to_datetime(cleaned["TEST DATE"], errors="coerce")
    bad_mask = dates.isna() | (dates.dt.year < 1900) | (dates.dt.year > 2030)
    if bad_mask.any():
        bad_files.append(p)
        print(f"{p.name}: bad {bad_mask.sum()} of {len(cleaned)}")
        display(
            cleaned.loc[
                bad_mask,
                ["source", "TEST DATE", "WELL", "TEST LENGHT IN HOURS"],
            ].head(5)
        )

print("bad files", len(bad_files))

for p in bad_files:
    engine = "xlrd" if p.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(p, engine=engine) as xl:
        sheet = mod.choose_sheet(xl)
        df = pd.read_excel(xl, sheet_name=sheet, header=None, dtype=str)

    df = df[df.iloc[:, mod.HEADER_FILTER_COL_INDEX].notna()]
    header_idx = mod.locate_header_index(df)
    if header_idx is not None:
        start = df.loc[header_idx:].copy()
    else:
        start = df.iloc[mod.SKIP_ROWS_AFTER_FILTER:].copy()

    header_row = start.iloc[0]
    subheader_row = start.iloc[1] if len(start) > 1 else None
    headers = mod.build_headers(header_row)
    if subheader_row is not None and mod.is_subheader_row(subheader_row):
        subheaders = mod.build_headers(subheader_row)
        headers = mod.merge_headers(headers, subheaders)

    normalized = [mod.normalize_col_name(c) for c in headers]
    mapped = [mod.COLUMN_ALIASES.get(name, name) for name in normalized]

    print(p.name, "header_idx", header_idx)
    print("date_cols", mod.find_date_columns(mapped))
    print("mapped head", mapped[:18])


02 FEB 2001 PARP REPORTS.xls: bad 243 of 533


C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date


,source,TEST DATE,WELL,TEST LENGHT IN HOURS
12,02 FEB 2001 PARP REPORTS.xls,NaT,QQ-001-32,<NA>
14,02 FEB 2001 PARP REPORTS.xls,NaT,A-069-32,<NA>
17,02 FEB 2001 PARP REPORTS.xls,NaT,OO-001-32,<NA>
19,02 FEB 2001 PARP REPORTS.xls,NaT,A-006-32,<NA>
21,02 FEB 2001 PARP REPORTS.xls,NaT,A-009-32,<NA>


C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date
C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date


05 MAY 2001 PARP REPORTS.xls: bad 315 of 534


...

10 OCT 2001 PARP REPORTS.xls: bad 211 of 518


,source,TEST DATE,WELL,TEST LENGHT IN HOURS
13,10 OCT 2001 PARP REPORTS.xls,NaT,A-069-32,<NA>
29,10 OCT 2001 PARP REPORTS.xls,NaT,A-029-32,<NA>
31,10 OCT 2001 PARP REPORTS.xls,NaT,A-033-32,<NA>
32,10 OCT 2001 PARP REPORTS.xls,NaT,A-034-32,<NA>
34,10 OCT 2001 PARP REPORTS.xls,NaT,A-036-32,<NA>


11 NOV 2001 PARP REPORTS.xls: bad 184 of 511


,source,TEST DATE,WELL,TEST LENGHT IN HOURS
13,11 NOV 2001 PARP REPORTS.xls,NaT,A-069-32,<NA>
28,11 NOV 2001 PARP REPORTS.xls,NaT,A-029-32,<NA>
30,11 NOV 2001 PARP REPORTS.xls,NaT,A-033-32,<NA>
31,11 NOV 2001 PARP REPORTS.xls,NaT,A-034-32,<NA>
33,11 NOV 2001 PARP REPORTS.xls,NaT,A-036-32,<NA>


bad files 6
02 FEB 2001 PARP REPORTS.xls header_idx 6
date_cols ('MTH', 'DAY', 'YR')
mapped head ['FIELD', 'RESVR', 'RESERVOIR', 'WELL', 'MTH', 'DAY', 'YR', 'HRs', 'WELL', 'PERC', 'GTY &', '64S', '****************W E L L T E S T D A T A ****************', 'TPR', 'SEP', 'BOPD', 'BWPD', 'GOR']
05 MAY 2001 PARP REPORTS.xls header_idx 6
date_cols ('MTH', 'DAY', 'YR')
mapped head ['FIELD', 'RESVR', 'RESERVOIR', 'WELL', 'MTH', 'DAY', 'YR', 'HRs', 'WELL', 'PERC', 'GTY &', '64S', '****************W E L L T E S T D A T A ****************', 'TPR', 'SEP', 'BOPD', 'BWPD', 'GOR']
08 AUG 2001 PARP REPORTS.xls header_idx 6
date_cols ('MTH', 'DAY', 'YR')
mapped head ['FIELD', 'RESVR', 'RESERVOIR', 'WELL', 'MTH', 'DAY', 'YR', 'HRs', 'WELL', 'PERC', 'GTY &', '64S', '****************W E L L T E S T D A T A ****************', 'TPR', 'SEP', 'BOPD', 'BWPD', 'GOR']
09 SEP 2001 PARP REPORTS.xls header_idx 6
date_cols ('MTH', 'DAY', 'YR')
mapped head ['FIELD', 'RESVR', 'RESERVOIR', 'WELL', 'MTH', 'DAY', 'YR', 

In [12]:
all_data = []
for p in files:
    cleaned = mod.transform_file(p)
    if not cleaned.empty:
        all_data.append(cleaned)

combined = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
print("rows", len(combined))

dates = pd.to_datetime(combined["TEST DATE"], errors="coerce")
bad = combined[dates.isna() | (dates.dt.year < 1900) | (dates.dt.year > 2030)]
print("bad dates", len(bad))
bad.head(20)

C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date
C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date
C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports.py:282: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["TEST DATE"] = pd.to_datetime(df["TEST DATE"], errors="coerce", dayfirst=True).dt.date
C:\Users\OPS010\Documents\projects\new QC\src\clean_parp_reports

rows 5861
bad dates 1364


,source,FIELD,RESVR,RESERVOIR,WELL,TEST DATE,TEST LENGHT IN HOURS,WELL TYPE,PCT WTR,GRAVITY,64S,PRES HEAD,PSIG SEP,DAILY OIL BBLS,DAILY WATER BBLS,GOR,GOR
554,02 FEB 2001 PARP REPORTS.xls,BAHI,BARG,BAHI-ARGUB SAND / STONE,QQ-001-32,NaT,NaN,NaN,61.8,44.4,0,NaN,NaN,NaN,NaN,33,...
555,02 FEB 2001 PARP REPORTS.xls,BAHI,BASS,BAHI SAND STONE,A-069-32,NaT,NaN,NaN,65.4,44.3,0,NaN,NaN,NaN,NaN,165,...
557,02 FEB 2001 PARP REPORTS.xls,BAHI,BAUK,BAHI UPER KALASH,OO-001-32,NaT,NaN,NaN,0,41.7,0,NaN,NaN,NaN,NaN,385,...
558,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-006-32,NaT,NaN,NaN,52.7,46.2,0,NaN,NaN,NaN,NaN,165,...
560,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-009-32,NaT,NaN,NaN,64.6,45.6,0,NaN,NaN,NaN,NaN,165,...
561,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-010-32,NaT,NaN,NaN,86.4,44.7,0,NaN,NaN,NaN,NaN,165,...
565,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-016-32,NaT,NaN,NaN,92.4,45.6,0,NaN,NaN,NaN,NaN,165,...
572,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-028-32,NaT,NaN,NaN,93.4,45.4,0,NaN,NaN,NaN,NaN,166,...
573,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-032-32,NaT,NaN,NaN,80.4,46.5,0,NaN,NaN,NaN,NaN,165,...
575,02 FEB 2001 PARP REPORTS.xls,BAHI,BPL7,PALEOCENE,A-034-32,NaT,NaN,NaN,93.8,45.6,0,NaN,NaN,NaN,NaN,165,...
